In [1]:
%%writefile search_engine.py
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

class InfoLocator:
    def __init__(self, dataset, method="tfidf"):
        if not dataset or any(not item["content"].strip() for item in dataset):
            raise ValueError("Dataset cannot be empty")
        self.dataset = dataset
        self.method = method
        corpus = [item["content"] for item in dataset]

        if method == "tfidf":
            self.vec = TfidfVectorizer()
            self.features = self.vec.fit_transform(corpus)
        elif method == "neural":
            from sentence_transformers import SentenceTransformer
            self.encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
            self.validate_len(corpus)
            self.features = self.encoder.encode(corpus, normalize_embeddings=True)
        else:
            raise ValueError("Invalid method. Use 'tfidf' or 'neural'")

    def validate_len(self, docs):
        for doc in docs:
            toks = len(self.encoder.tokenizer(doc, truncation=False)["input_ids"])
            if toks > self.encoder.max_seq_length:
                raise ValueError("Text too long for the encoder")

    def find(self, user_query, top_n=3, criteria=None):
        if not user_query.strip() or top_n <= 0:
            return []

        if self.method == "tfidf":
            q_vec = self.vec.transform([user_query])
            if q_vec.nnz == 0: return []
            similarities = (q_vec @ self.features.T).toarray()[0]
        else:
            self.validate_len([user_query])
            q_vec = self.encoder.encode([user_query], normalize_embeddings=True)[0]
            similarities = self.features @ q_vec

        valid_indices = [idx for idx, row in enumerate(self.dataset)
                         if all(row.get(k) == v for k, v in (criteria or {}).items())]

        sorted_idx = sorted(valid_indices, key=lambda idx: (-float(similarities[idx]), idx))
        return [(self.dataset[idx], float(similarities[idx])) for idx in sorted_idx[:top_n]]

def display_results(results):
    if not results:
        print("No results found.")
    for doc, sim in results:
        print(doc["id"], f"confidence={sim:.3f}")
        print(doc["content"])

Writing search_engine.py


In [2]:
%%writefile customer_bot.py
from search_engine import InfoLocator

KB_FAQS = [
    {"id": "faq:delivery", "q": "Do you offer same-day delivery?",
     "a": "Currently, we only offer next-day delivery for premium members.", "ref": "shipping_guidelines.txt"},
    {"id": "faq:refund", "q": "How long does a refund take?",
     "a": "Refunds are processed within 5-7 business days after receiving the returned item.", "ref": "billing_info.txt"},
    {"id": "faq:warranty", "q": "Are physical damages covered?",
     "a": "No, standard warranty only covers manufacturing defects.", "ref": "warranty_terms.txt"},
    {"id": "faq:support", "q": "How can I speak to a human agent?",
     "a": "You can call our toll-free number during business hours.", "ref": "contact_us.txt"},
    {"id": "faq:internals", "q": "Will opening the device void my warranty?",
     "a": "Yes, breaking the factory seal voids the warranty completely.", "ref": "warranty_terms.txt"}
]

def init_assistant(method="tfidf"):
    dataset = [dict(item, content=item["q"] + " " + item["a"]) for item in KB_FAQS]
    return InfoLocator(dataset, method)

def generate_response(system, user_input):
    if not user_input.strip():
        return "Awaiting your query."

    for item in KB_FAQS:
        if user_input.strip().lower() == item["q"].lower():
            return f"{item['a']} [reference: {item['ref']}]"

    matches = system.find(user_input)
    if not matches:
        return "I couldn't find an answer. Please reach out to support@company.com."

    resp = ["Here are some relevant topics you might be asking about:"]
    for item, sim in matches:
        resp.append(f"- {item['q']} (confidence {sim:.2f})")
    return "\n".join(resp)

if __name__ == "__main__":
    import argparse
    p = argparse.ArgumentParser()
    p.add_argument("--method", choices=["tfidf", "neural"], default="tfidf")
    args = p.parse_args()

    sys = init_assistant(args.method)
    print("Welcome to Customer Support. Type 'quit' to exit.")
    while True:
        try:
            user_input = input("Customer: ")
        except (EOFError, KeyboardInterrupt):
            break
        if user_input.strip().lower() == "quit":
            break
        print("Agent:", generate_response(sys, user_input))

Writing customer_bot.py


In [3]:
!ls

customer_bot.py  sample_data  search_engine.py


In [4]:
%%writefile text_processor.py
import re

def get_word_spans(doc):
    return [(match.start(), match.end()) for match in re.finditer(r"\S+", doc)]

def sliding_blocks(doc, max_words=50, overlap_words=15, start_offset=0):
    if max_words <= 0 or not 0 <= overlap_words < max_words:
        raise ValueError("Invalid parameters for sliding blocks")

    w_spans = get_word_spans(doc)
    blocks = []
    step = max_words - overlap_words

    for idx in range(0, len(w_spans), step):
        end_idx = min(idx + max_words, len(w_spans))
        blocks.append((start_offset + w_spans[idx][0], start_offset + w_spans[end_idx - 1][1]))
        if end_idx == len(w_spans):
            break
    return blocks

def smart_sentences(doc, max_words=50):
    if max_words <= 0:
        raise ValueError("max_words must be > 0")

    boundaries = [0] + [match.end() for match in re.finditer(r"[.!?]\s+", doc)] + [len(doc)]
    segments, buffer, word_count = [], None, 0

    for start, end in zip(boundaries, boundaries[1:]):
        text_slice = doc[start:end]
        slice_len = len(text_slice.split())
        if not slice_len: continue

        if slice_len > max_words:
            if buffer:
                segments.append(buffer)
            segments.extend(sliding_blocks(text_slice, max_words, 0, start))
            buffer, word_count = None, 0
        elif buffer and word_count + slice_len <= max_words:
            buffer = (buffer[0], end)
            word_count += slice_len
        else:
            if buffer:
                segments.append(buffer)
            buffer = (start, end)
            word_count = slice_len

    if buffer:
        segments.append(buffer)
    return segments

def chunk_documents(corpus, mode):
    chunks = []
    for doc_key, content in corpus.items():
        if mode == "smart":
            spans = smart_sentences(content)
        else:
            spans = sliding_blocks(content, 50, int(mode))

        for idx, (s, e) in enumerate(spans):
            chunks.append(dict(id=f"{doc_key}_part{idx}", src_doc=doc_key, start_pos=s, end_pos=e, content=content[s:e]))
    return chunks

if __name__ == "__main__":
    assert sliding_blocks("") == []
    assert sliding_blocks("hello world") == [(0, 11)]
    try:
        sliding_blocks("a b", 2, 2)
    except ValueError:
        pass
    test_str = " ".join(["test"] * 120) + "."
    assert all(len(test_str[s:e].split()) <= 50 for s, e in smart_sentences(test_str))
    test_corpus = {"doc1": "Welcome to our platform. Ensure your data is safe."}
    for m in ["0", "15", "smart"]:
        for ch in chunk_documents(test_corpus, m):
            assert ch["content"] == test_corpus[ch["src_doc"]][ch["start_pos"]:ch["end_pos"]]

Writing text_processor.py


In [5]:
!python text_processor.py

In [6]:
%%writefile test_processor.py
from text_processor import sliding_blocks, smart_sentences
sample = ("Returns are accepted within thirty days. Products must be unboxed and "
          "in original packaging to qualify for a full refund. Contact support.")
print("--- Sliding Blocks ---")
for s, e in sliding_blocks(sample, max_words=10, overlap_words=3):
    print(s, e, repr(sample[s:e]))
print("\n--- Smart Sentences ---")
for s, e in smart_sentences(sample, max_words=15):
    print(s, e, repr(sample[s:e]))

Writing test_processor.py


In [7]:
!python test_processor.py

--- Sliding Blocks ---
0 65 'Returns are accepted within thirty days. Products must be unboxed'
50 106 'must be unboxed and in original packaging to qualify for'
92 138 'to qualify for a full refund. Contact support.'

--- Smart Sentences ---
0 41 'Returns are accepted within thirty days. '
41 122 'Products must be unboxed and in original packaging to qualify for a full refund. '
122 138 'Contact support.'


In [8]:
%%writefile assess_docs.py
from text_processor import chunk_documents, sliding_blocks, smart_sentences

DOC_CORPUS = {
    "shipping": (
        "Logistics Protocol. All orders are dispatched within two business days "
        "from our primary warehouse. Standard delivery typically takes between "
        "three to five business days depending on your location. Express options "
        "are available at checkout for an additional fee. If a package is marked "
        "as delivered but cannot be found, customers must wait twenty-four hours "
        "before filing a claim, as carriers sometimes scan items early. International "
        "shipments may be subject to customs duties which are the responsibility "
        "of the buyer. In case of extreme weather, deliveries might be delayed "
        "without prior notice. Always ensure your tracking number is kept secure "
        "until the package is firmly in your possession."
    ),
    "returns": (
        "Return and Refund Policy. Customers can initiate a return request within "
        "thirty days of receiving their product. The item must remain in its original "
        "factory condition with all seals intact. Used items are subject to a restocking "
        "fee of up to twenty percent. To start the process, log into your account and "
        "generate a return merchandise authorization label. Print this label and attach "
        "it clearly to the outside of the box. Drop the package off at any certified "
        "postal facility. Once we receive the warehouse scan, your refund will be "
        "processed back to the original payment method within five to seven business "
        "days. Exchanges are treated as a separate return and new purchase transaction."
    ),
    "warranty": (
        "Hardware Warranty terms. Our electronic devices come with a standard one year "
        "limited warranty covering manufacturer defects and internal component failures. "
        "This warranty does not cover accidental damage, liquid spills, or unauthorized "
        "third-party repairs. If you experience a malfunction, contact our technical "
        "support team to troubleshoot the issue remotely before sending the unit in. "
        "Should a replacement be necessary, we reserve the right to provide a certified "
        "refurbished unit of equal or greater value. Customers are responsible for the "
        "inbound shipping costs for warranty claims, while we cover the return shipping. "
        "Please back up all personal data before mailing your device, as we wipe all "
        "storage drives to protect customer privacy."
    )
}

if __name__ == "__main__":
    stats = {d: len(t.split()) for d, t in DOC_CORPUS.items()}
    for mode, desc in [("0", "Sliding 50, No overlap"),
                       ("15", "Sliding 50, Overlap 15"),
                       ("smart", "Smart Sentences 50")]:
        all_chunks = chunk_documents(DOC_CORPUS, mode)
        print(f"\nMethod: {desc}")
        print(f"Total chunks generated: {len(all_chunks)}")
        for ch in all_chunks[:3]:
            print("  ", ch["id"], "->", len(ch["content"].split()), "words")

Writing assess_docs.py


In [9]:
!python assess_docs.py


Method: Sliding 50, No overlap
Total chunks generated: 9
   shipping_part0 -> 50 words
   shipping_part1 -> 50 words
   shipping_part2 -> 7 words

Method: Sliding 50, Overlap 15
Total chunks generated: 9
   shipping_part0 -> 50 words
   shipping_part1 -> 50 words
   shipping_part2 -> 37 words

Method: Smart Sentences 50
Total chunks generated: 9
   shipping_part0 -> 38 words
   shipping_part1 -> 41 words
   shipping_part2 -> 28 words


In [10]:
%%writefile test_neural.py
import numpy as np
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
texts = [
    "Initiate a return request within thirty days.",
    "International shipments may be subject to customs duties.",
    "Hardware warranty covers manufacturer defects only."
]
vectors = encoder.encode(texts, normalize_embeddings=True)
q_vec = encoder.encode(["Do I pay for customs?"], normalize_embeddings=True)

similarities = (q_vec @ vectors.T)[0]
print("Vector dimensions:", vectors.shape)
for idx in np.argsort(-similarities):
    print(f"Rank {idx}: {float(similarities[idx]):.3f} | {texts[idx]}")

Writing test_neural.py


In [11]:
!python test_neural.py

modules.json: 100% 349/349 [00:00<00:00, 1.32MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 356kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 19.1MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 209kB/s]
config.json: 100% 612/612 [00:00<00:00, 2.67MB/s]

model.safetensors: downloading bytes:  94% 85.0M/90.9M [00:01<00:00, 97.3MB/s, 7.62MB/s  ]
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:01<00:00, 70.8MB/s, 8.12MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:01<00:00, 75.6MB/s, 8.84MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 3789.46it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.29MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 13.7MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 40.6MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 489kB/s]
config.json: 100% 190/190 [00:00<00:00, 614kB/s]
Vector dimensions: (3, 384)
Rank 1: 0.693 | International shipments may be subject to customs duties.
R

In [12]:
%%writefile agent_suggestions.py
from customer_bot import init_assistant
agent = init_assistant("neural")
user_query = "item broken warranty"
print("Did you mean:")
for res, conf in agent.find(user_query, top_n=3):
    print(f"- {res['q']} (Confidence: {conf:.2f})")

Writing agent_suggestions.py


In [13]:
!python agent_suggestions.py

Loading weights: 100% 103/103 [00:00<00:00, 13971.52it/s]
Did you mean:
- Are physical damages covered? (Confidence: 0.51)
- Will opening the device void my warranty? (Confidence: 0.46)
- How long does a refund take? (Confidence: 0.17)


In [14]:
%%writefile semantic_eval.py
from text_processor import chunk_documents
from assess_docs import DOC_CORPUS
from search_engine import InfoLocator

corpus_data = chunk_documents(DOC_CORPUS, "smart")
test_queries = [
    "Who pays for the customs tax?",
    "How do I send my device back for refund?",
    "Will a water spill be covered?"
]

for method in ["tfidf", "neural"]:
    print(f"\n{'='*40}\nEvaluating Method: {method.upper()}")
    system = InfoLocator(corpus_data, method)
    for tq in test_queries:
        print(f"\nQuery: {tq}")
        results = system.find(tq, top_n=2)
        for res, conf in results:
            print(f"  [{res['id']}] Score: {conf:.3f}\n    {res['content']}")

Writing semantic_eval.py


In [15]:
!python semantic_eval.py


Evaluating Method: TFIDF

Query: Who pays for the customs tax?
  [warranty_part2] Score: 0.231
    Customers are responsible for the inbound shipping costs for warranty claims, while we cover the return shipping. Please back up all personal data before mailing your device, as we wipe all storage drives to protect customer privacy.
  [shipping_part1] Score: 0.180
    If a package is marked as delivered but cannot be found, customers must wait twenty-four hours before filing a claim, as carriers sometimes scan items early. International shipments may be subject to customs duties which are the responsibility of the buyer. 

Query: How do I send my device back for refund?
  [warranty_part2] Score: 0.305
    Customers are responsible for the inbound shipping costs for warranty claims, while we cover the return shipping. Please back up all personal data before mailing your device, as we wipe all storage drives to protect customer privacy.
  [returns_part2] Score: 0.165
    Once we receive t